In [2]:
!pip install fastembed

  Using cached fastembed-0.8.0-py3-none-any.whl.metadata (10 kB)
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached py_rust_stemmers-0.1.8-cp312-cp312-win_amd64.whl.metadata (3.6 kB)
  Using cached win32_setctime-1.2.0-py3-none-any.whl.metadata (2.4 kB)
Using cached fastembed-0.8.0-py3-none-any.whl (116 kB)
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.2 MB 3.4 MB/s eta 0:00:02
   ---------- ----------------------------- 1.8/7.2 MB 4.4 MB/s eta 0:00:02
   ------------------ --------------------- 3.4/7.2 MB 5.4 MB/s eta 0:00:01
   --------------------- ------------------ 3.9/7.2 MB 5.6 MB/s eta 0:00:01
   ------------------------------- -------- 5.8/7.2 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 5.8 MB/s  0:00:01
Using cached py_rust_st

In [3]:
import fastembed
print("fastembed installed successfully")

fastembed installed successfully


In [4]:
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Old storage removed")

from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import re

with open("telecom_faq_knowledge_base.md", "r", encoding="utf-8") as f:
    raw_text = f.read()

raw_chunks = re.split(r"\n### ", raw_text)
qa_chunks = ["### " + chunk.strip() for chunk in raw_chunks[1:]]
documents = [Document(page_content=chunk) for chunk in qa_chunks]

embedding_model = FastEmbedEmbeddings(model_name="intfloat/multilingual-e5-small")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print(f"Done — rebuilt storage with {len(documents)} chunks")

ValidationError: 1 validation error for FastEmbedEmbeddings
  Value error, Model intfloat/multilingual-e5-small is not supported in TextEmbedding. Please check the supported models using `TextEmbedding.list_supported_models()` [type=value_error, input_value={'model_name': 'intfloat/...s': None, 'model': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [5]:
from fastembed import TextEmbedding

models = TextEmbedding.list_supported_models()
for m in models:
    if "multilingual" in m["model"].lower() or "e5" in m["model"].lower():
        print(m["model"])

sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
sentence-transformers/paraphrase-multilingual-mpnet-base-v2
intfloat/multilingual-e5-large


In [6]:
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Old storage removed")

from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import re

with open("telecom_faq_knowledge_base.md", "r", encoding="utf-8") as f:
    raw_text = f.read()

raw_chunks = re.split(r"\n### ", raw_text)
qa_chunks = ["### " + chunk.strip() for chunk in raw_chunks[1:]]
documents = [Document(page_content=chunk) for chunk in qa_chunks]

embedding_model = FastEmbedEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print(f"Done — rebuilt storage with {len(documents)} chunks")

C:\Users\uzair\anaconda3\envs\lc-academy-env\Lib\site-packages\langchain_community\embeddings\fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

C:\Users\uzair\anaconda3\envs\lc-academy-env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\uzair\AppData\Local\Temp\fastembed_cache\models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Done — rebuilt storage with 50 chunks


In [7]:
results = vectorstore.similarity_search("What happens if I miss a payment deadline?", k=3)
for doc in results:
    print(doc.page_content)
    print("---")

### Q6 (EN): What happens if I miss a payment deadline?
A late payment fee of AED 25 applies after a 5-day grace period. If payment remains outstanding for more than 15 days, outgoing services may be suspended. Full service suspension occurs after 30 days of non-payment.
---
### Q6 (AR): ماذا يحدث إذا فاتني موعد السداد؟
تُطبّق رسوم تأخير قدرها 25 درهمًا بعد فترة سماح مدتها 5 أيام. وإذا استمر عدم السداد لأكثر من 15 يومًا، قد يتم إيقاف الخدمات الصادرة. ويتم إيقاف الخدمة بالكامل بعد 30 يومًا من عدم السداد.
---
### Q7 (AR): كيف يمكنني الاعتراض على رسوم في فاتورتي؟
يمكنك تقديم اعتراض على الفاتورة من خلال التطبيق عبر "الدعم" ثم "مشكلة في الفاتورة"، أو بالاتصال بخدمة العملاء، أو عبر الدردشة المباشرة. يتم حل معظم الاعتراضات خلال 5 إلى 7 أيام عمل، وسيتم رد أي رسوم خاطئة أو إضافتها كرصيد في فاتورتك القادمة.

---

## Category: Roaming
---


In [8]:
requirements_content = """chainlit
langchain
langchain-groq
langchain-chroma
langchain-community
fastembed
python-dotenv
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

print("requirements.txt updated")

requirements.txt updated
